In [2]:
# Environment setup -- do NOT pip install from inside the notebook.
#
#     pip install -r requirements.txt
#
# requirements.txt pins torch 2.9.1+cu130 with the matching torchaudio and
# torchcodec, plus prebuilt mamba-ssm / causal-conv1d wheels for cu13+torch2.9.
from python_files.DataLoaders import *

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from python_files.DataLoaders import *
from python_files.MambaBlocks import *
from python_files.VectorQuantizer import *
from python_files.SpeakerInjector import *

##Used only for training
from python_files.SpeakerAdversary import *

from speechbrain.inference.speaker import EncoderClassifier

from speechbrain.inference.vocoders import HIFIGAN

import os

import torchaudio

# --- CUDA setup -------------------------------------------------------------
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True          # shapes are bounded by max_frames
torch.set_float32_matmul_precision("high")


def normalize_device(device):
    """'cuda' -> 'cuda:N'.

    SpeechBrain resolves a device string by splitting on ':' and unpacking two
    values, so an index-less 'cuda' raises before it falls back to device 0.
    """
    d = torch.device(device)
    if d.type == "cuda" and d.index is None:
        d = torch.device(f"cuda:{torch.cuda.current_device()}")
    return d


ModuleNotFoundError: No module named 'mamba_ssm'

In [ ]:
## CONSTANTS

N_MELS    = 80
D_STATE   = 16
D_MODEL   = 256
D_CONV    = 4
EXPAND    = 2
N_LAYERS  = 6
SPK_DIM   = 192   # raw ECAPA-TDNN output, no projection
# Codebook for new tokenizers (pretrain / tokenizer modes). Modes that reuse a frozen codebook
# (codes2mel, recognizer, rec2mel) and the jasper loaders read the sizes from the checkpoint.
VQ_CODES  = 512
DROPOUT   = 0.1
VQ_GROUPS = 4
CODE_DIM  = 32
SKIP_DIM  = 64


## Configuration


MODE_CONFIG = {
    "pretrain": {
        "lr": 3e-4,
        "batch_size": 16,
        "epochs": 50,
        "warmup_steps": 5000,
        "grad_clip": 1.0,
        "log_interval": 50,
        "eval_interval": 2000,
        "save_interval": 5000,
    },
    "finetune": {
        "lr": 2e-5,
        "batch_size": 8,
        "epochs": 30,
        "warmup_steps": 1000,
        "grad_clip": 1.0,
        "log_interval": 20,
        "eval_interval": 500,
        "save_interval": 2000,
    },
    "e2e": {
        "lr": 1e-5,
        "batch_size": 8,
        "epochs": 10,
        "warmup_steps": 0,
        "grad_clip": 0.5,
        "log_interval": 20,
        "eval_interval": 500,
        "save_interval": 1000,
    },
    # Code-prediction design, step 1: re-train the whole core with NO skip path, so the VQ codes
    # must carry the content. LibriTTS + UA control speakers (Codes2MelDataset), B2 excluded.
    "tokenizer": {
        "adv_weight": 0.0,             # speaker adversary off (its CE spikes destabilised training)
        # Consistency loss: pull the pre-quantisation latents of two recordings of the same word together
        # (another mic of the same take, or another control speaker / block), DTW-aligned.
        # 0 = off (plain reconstruction tokenizer).
        "consistency_weight": 0.1,
        "lr": 3e-4,
        "batch_size": 16,
        "epochs": 15,
        "warmup_steps": 2000,
        "grad_clip": 1.0,
        "log_interval": 50,
        "eval_interval": 1000,
        "save_interval": 2000,
    },
    # Code-prediction design, step 2: dysarthric mel -> encoder -> 4 heads -> CF02's codes (cross-entropy).
    # VQ codebook, injector and decoder come frozen from the tokenizer checkpoint.
    "recognizer": {
        "lr": 2e-4,
        "batch_size": 16,
        "epochs": 15,
        "warmup_steps": 1000,
        "grad_clip": 1.0,
        "log_interval": 50,
        "eval_interval": 500,
        "save_interval": 2000,
        "head_layers": 2,               # extra bidirectional Mamba layers between encoder and code heads
        # multi_ref: average the targets of several reference takes (CF02's other block + CF03/CF04/CF05),
        # so codes the references agree on dominate.
        "multi_ref": True,              # needs precompute_multi_target_codes(...) once; False = CF02 only
        # merge_classes: train on merged code classes (summed code probabilities); the decoder is fed each
        # class's most-used code. Pick the size with oracle_codes_check(merge_classes=...) in jasper.ipynb.
        "merge_classes": None,          # e.g. 256; None = raw codes
        "soft_tau": 0.5,                # distance-aware soft targets (smaller = sharper); 0 = one-hot CE
        "head_dropout": 0.1,            # dropout before the code heads
        "freeze_lower_encoder": False,  # whole encoder learns (True = input_proj, layers 1-3, downsample frozen)
        "specaug": True,                # time/frequency masking on the input mel (training only)
    },
    # Code-prediction step 3: the decoder learns to speak from the recognizer's predicted codes.
    # Recognizer and codebook frozen; injector + decoder learn, masked mel L1 against the DTW-aligned
    # CF02 target. Start from the recognizer checkpoint.
    "rec2mel": {
        "lr": 1e-4,
        "batch_size": 16,
        "epochs": 8,
        "warmup_steps": 500,
        "grad_clip": 1.0,
        "log_interval": 50,
        "eval_interval": 500,
        "save_interval": 2000,
        "specaug": False,      # the recognizer is frozen; feed it clean inputs
    },
    # Decoder-only variant: learns mel from a FROZEN tokenizer's codes (needed only if the tokenizer
    # was trained with skip; a no-skip tokenizer's decoder already speaks from codes)
    "codes2mel": {
        "lr": 2e-4,
        "batch_size": 16,
        "epochs": 10,
        "warmup_steps": 500,
        "grad_clip": 1.0,
        "log_interval": 50,
        "eval_interval": 1000,
        "save_interval": 2000,
    },
    "vocoder_adapt": {
        "lr": 1e-5,
        "batch_size": 16,
        "epochs": 20,
        "warmup_steps": 0,
        "grad_clip": 5.0,
        "log_interval": 50,
        "eval_interval": 1000,
        "save_interval": 2000,
    },
}

In [ ]:
"""
Full speech reconstruction pipeline.

Wires together: Encoder → VQ → Skip → Adversary → SpeakerInjector → Decoder
Pretrained: SpeechBrain ECAPA-TDNN (speaker) + HiFi-GAN (vocoder)
"""

import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from speechbrain.inference.speaker import EncoderClassifier
from speechbrain.inference.vocoders import HIFIGAN





def normalize_device(device):
    return torch.device(device)


class SpeechReconstructionPipeline(nn.Module):
    

    def __init__(
        self,
        d_model=D_MODEL,
        d_state=D_STATE,
        d_conv=D_CONV,
        expand=EXPAND,
        n_layers=N_LAYERS,
        spk_dim=SPK_DIM,
        n_mels=N_MELS,
        vq_groups=VQ_GROUPS,
        vq_codes=VQ_CODES,
        code_dim=CODE_DIM,
        skip_dim=SKIP_DIM,
        speaker_adv_weight=0.1,
        num_speakers=100,
        dropout=DROPOUT,
        head_layers=0,
        merge_classes=0,
        device="cuda:0",
    ):
        super().__init__()

        self.dev = normalize_device(device)
        self.speaker_adv_weight = speaker_adv_weight
        self._resamplers = {}
        self.mode = None
        # Skip path (encoder_out -> 64 dims, bypassing the VQ) is off: it let the decoder ignore the codes.
        # True only to evaluate old skip checkpoints.
        self.use_skip = False
        # Must match preprocess.py's mel_transform. Wrapped in a list so it stays out of state_dict.
        self._wav_to_mel = [torchaudio.transforms.MelSpectrogram(
            sample_rate=22050, n_fft=1024, hop_length=256, win_length=1024, n_mels=n_mels,
            f_min=0.0, f_max=8000.0, power=2.0, norm="slaney", mel_scale="slaney",
        )]

        run_opts = {"device": str(self.dev)}

        # --- Frequency-weighted L1 loss ---
        # Bins 0→79 get weights 1.0→3.0 (high freq penalized more)
        freq_weights = torch.linspace(1.0, 3.0, n_mels).view(1, n_mels, 1)
        self.register_buffer("freq_weights", freq_weights)

        # --- Pretrained: ECAPA-TDNN speaker encoder (frozen) ---
        self.speaker_encoder = EncoderClassifier.from_hparams(
            source="speechbrain/spkrec-ecapa-voxceleb",
            savedir=os.path.join(Path.home(), ".cache", "speechbrain", "spkrec-ecapa"),
            run_opts=run_opts,
        )
        for param in self.speaker_encoder.mods.parameters():
            param.requires_grad = False

        # --- Pretrained: HiFi-GAN vocoder (frozen) ---
        self.vocoder = HIFIGAN.from_hparams(
            source="speechbrain/tts-hifigan-ljspeech",
            savedir=os.path.join(Path.home(), ".cache", "speechbrain", "tts-hifigan-ljspeech"),
            run_opts=run_opts,
        )
        for param in self.vocoder.mods.parameters():
            param.requires_grad = False

        # --- Core model ---
        self.encoder = BidirectionalMambaEncoder(
            n_mels=n_mels, d_model=d_model, d_state=d_state,
            d_conv=d_conv, expand=expand, dropout=dropout, n_layers=n_layers,
        )

        projected_dim = vq_groups * code_dim

        self.vq = ProductQuantizer(
            input_dim=d_model, num_groups=vq_groups, code_dim=code_dim,
            commitment_weight=1.0, entropy_weight=0.5,
            num_codes=vq_codes, decay=0.9, reset_thresh=2,
        )

        self.skip_proj = nn.Linear(d_model, skip_dim)

        self.speaker_adversary = SpeakerAdversary(
            input_dim=projected_dim, hidden_dim=128,
            num_speakers=num_speakers, grl_scale=1.0,
        )

        self.speaker_injector = SpeakerInjector(
            vq_dim=projected_dim + skip_dim,  # VQ + skip concatenated
            spk_dim=spk_dim,                   # raw ECAPA 192-dim
            d_model=d_model,
        )

        self.decoder = CausalMambaDecoder(
            n_mels=n_mels, d_model=d_model, d_state=d_state,
            d_conv=d_conv, expand=expand, dropout=dropout, n_layers=n_layers,
        )

        # --- Recognizer (code prediction) ---
        # One classifier per VQ group: encoder frame -> which of the 512 codes CF02 would use here.
        self.code_heads = nn.ModuleList([nn.Linear(d_model, vq_codes) for _ in range(vq_groups)])
        # Extra bidirectional Mamba layers between the encoder and the heads (0 = heads read the encoder
        # output directly). The count is read back from checkpoints.
        self.code_trunk = nn.ModuleList([
            BirectionalMambaLayer(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand, dropout=dropout)
            for _ in range(head_layers)
        ])
        self.code_trunk_norm = nn.RMSNorm(d_model) if head_layers > 0 else nn.Identity()
        # Merged code classes (code_merge_map): code -> class per group, and the code fed to the decoder
        # for each class. Saved in the checkpoint; size read back by codebook_arch. None = raw codes.
        if merge_classes > 0:
            self.register_buffer("merge_cls", torch.zeros(vq_groups, vq_codes, dtype=torch.long))
            self.register_buffer("merge_rep", torch.zeros(vq_groups, merge_classes, dtype=torch.long))
        else:
            self.merge_cls = self.merge_rep = None
        self.predict_codes = False      # True: decode from predicted codes (recognizer mode / checkpoints)
        # Voice the decoder speaks in when predicting codes (CF02, copied from the tokenizer checkpoint).
        self.register_buffer("reference_emb", torch.zeros(spk_dim))
        self.head_dropout = nn.Dropout(0.0)   # p set from MODE_CONFIG in train()
        self.soft_tau = 0.0                   # >0: soft targets over acoustically close codes
        self.consistency_weight = 0.0         # weight of the cross-recording latent loss
        # How predicted codes become decoder input (recognizer inference; training always uses "argmax"):
        #   "argmax"      each frame takes its most likely code
        #   "smooth"      average log-probs over +-smooth_width frames first, then argmax
        #   "soft"        probability-weighted mix of code embeddings instead of one code
        #   "smooth_soft" both
        self.decode = "argmax"
        self.smooth_width = 2

    # Speaker embedding (raw ECAPA, no projection)
    def extract_speaker_embedding(self, waveform, sample_rate=22050):
        """(B, samples) → (B, 192) L2-normalized speaker embedding."""
        if sample_rate != 16000:
            key = int(sample_rate)
            if key not in self._resamplers:
                self._resamplers[key] = torchaudio.transforms.Resample(
                    key, 16000
                ).to(waveform.device)
            waveform = self._resamplers[key](waveform)

        with torch.no_grad():
            raw = self.speaker_encoder.encode_batch(waveform).squeeze(1)

        return F.normalize(raw, p=2, dim=-1)

    # Vocoder
    def mel_to_waveform(self, mel):
        """(B, 80, T) mel → (B, 1, samples) waveform."""
        with torch.no_grad():
            wav = self.vocoder.decode_batch(mel)
        return wav.unsqueeze(1) if wav.dim() == 2 else wav

    # Codes -> VQ output (recognizer)
    def merge_onehot(self, g, dtype=torch.float32):
        """(codes, classes) 0/1 matrix for group g: code probabilities @ it = class probabilities."""
        return F.one_hot(self.merge_cls[g], self.merge_rep.shape[1]).to(dtype)

    def codes_to_quantized(self, codes):
        """(B, T', G) code indices -> (B, T', G*code_dim) exactly as the VQ would output them."""
        parts = [F.embedding(codes[..., g], q.embedding) for g, q in enumerate(self.vq.quantizers)]
        return self.vq.post_proj(torch.cat(parts, dim=-1))

    def probs_to_quantized(self, probs):
        """(B, T', G, K) code probabilities -> expected code embedding per group, then post_proj."""
        parts = [probs[..., g, :] @ q.embedding for g, q in enumerate(self.vq.quantizers)]
        return self.vq.post_proj(torch.cat(parts, dim=-1))

    def smooth_logprobs(self, code_logits, width):
        """Average log-probabilities over +-width frames (per group, per code)."""
        lp = F.log_softmax(code_logits, dim=-1)                          # (B,T',G,K)
        B, T, G, K = lp.shape
        x = lp.permute(0, 2, 3, 1).reshape(B, G * K, T)
        x = F.avg_pool1d(x, kernel_size=2 * width + 1, stride=1, padding=width, count_include_pad=False)
        return x.reshape(B, G, K, T).permute(0, 3, 1, 2)

    # Training mode
    def train(self, mode=True):
        super().train(mode)
        # vocoder_adapt: keep the frozen core in eval so VQ EMA / dead-code resets
        # don't mutate the codebook while only the vocoder is training.
        if mode and self.mode == "vocoder_adapt":
            for m in [self.encoder, self.vq, self.skip_proj,
                      self.speaker_adversary, self.speaker_injector, self.decoder]:
                m.eval()
        # codes2mel: encoder + VQ are a frozen tokenizer; no dropout, no EMA / dead-code resets
        if mode and self.mode == "codes2mel":
            for m in [self.encoder, self.vq, self.skip_proj, self.speaker_adversary]:
                m.eval()
        # recognizer: only the encoder (+ heads) learns; the codebook and decoder must not change
        if mode and self.mode == "recognizer":
            for m in [self.vq, self.skip_proj, self.speaker_adversary, self.speaker_injector, self.decoder]:
                m.eval()
        # rec2mel: the recognizer is frozen too (no dropout), so the decoder trains on exactly the
        # codes it will get at inference; only injector + decoder are in train mode
        if mode and self.mode == "rec2mel":
            for m in [self.encoder, self.code_trunk, self.code_trunk_norm, self.code_heads, self.head_dropout,
                      self.vq, self.skip_proj, self.speaker_adversary]:
                m.eval()
        return self

    def set_mode(self, mode):
        """
        Freeze/unfreeze for training phases.
        ECAPA and vocoder are always frozen (unfrozen only in vocoder_adapt).
        """
        self.mode = mode
        self.predict_codes = mode in ("recognizer", "rec2mel")
        for param in self.parameters():
            param.requires_grad = False

        core = [
            self.encoder, self.vq, self.skip_proj,
            self.speaker_adversary, self.speaker_injector, self.decoder,
        ]

        if mode in ("pretrain", "tokenizer"):
            for m in core:
                for param in m.parameters():
                    param.requires_grad = True

        elif mode == "finetune":
            # Freeze encoder layers below downsample
            for i, layer in enumerate(self.encoder.layers):
                if i >= self.encoder.downsample_after:
                    for param in layer.parameters():
                        param.requires_grad = True
            # Unfreeze everything else in core
            for m in [self.encoder.final_norm, self.vq, self.skip_proj,
                      self.speaker_adversary, self.speaker_injector, self.decoder]:
                for param in m.parameters():
                    param.requires_grad = True

        elif mode == "e2e":
            for m in core:
                for param in m.parameters():
                    param.requires_grad = True

        elif mode == "codes2mel":
            # Tokenizer (encoder + VQ) frozen; injector + decoder learn to speak from codes alone.
            for m in [self.speaker_injector, self.decoder]:
                for param in m.parameters():
                    param.requires_grad = True

        elif mode == "rec2mel":
            # Recognizer frozen; injector + decoder learn to speak from its predicted codes.
            for m in [self.speaker_injector, self.decoder]:
                for param in m.parameters():
                    param.requires_grad = True

        elif mode == "recognizer":
            # Encoder learns to map dysarthric speech to CF02's codes; everything after the codes is frozen.
            for m in [self.encoder, self.code_trunk, self.code_trunk_norm, self.code_heads]:
                for param in m.parameters():
                    param.requires_grad = True

        elif mode == "vocoder_adapt":
            for param in self.vocoder.mods.parameters():
                param.requires_grad = True
            # decode_batch strips weight norm on its first call, which would orphan the
            # weight_g/weight_v params the optimizer holds. Keep weight norm on.
            self.vocoder.first_call = False

        else:
            raise ValueError(f"Unknown mode: {mode}")

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"Mode: {mode} — {trainable:,} / {total:,} params trainable")

    # Adversary scale
    def set_adversary_scale(self, step, ramp_steps=50000):
        """Ramp GRL from 0 → max over ramp_steps."""
        adv = self.speaker_adversary
        adv.set_current_scale(adv.grl_scale * min(1.0, step / ramp_steps))

    # Forward
    def forward(self, x, padding_mask, speaker_ids, speaker_emb,
                target=None, run_vocoder=False, target_codes=None,
                pair_mel=None, pair_padding_mask=None, pair_index=None):
        """
        Args:
            x:            (B, 80, T) input mel
            padding_mask: (B, T) True where padded
            speaker_ids:  (B,) integer labels for adversary
            speaker_emb:  (B, 192) speaker embedding
            target:       (B, 80, T) target mel (None at inference)
            run_vocoder:  generate waveform (skip during training)
            target_codes: (B, T', G) CF02 code indices, -100 = padding (recognizer only)
            pair_mel, pair_padding_mask, pair_index: second recording of the same word and the
                          DTW frame map input -> pair (-1 = no pair); tokenizer consistency loss only

        Returns:
            dict with predicted, loss, mel_loss, vq_loss, adv_loss, vq_op, predicted_waveform
        """
        # 1. Encode
        encoder_out = self.encoder(x, padding_mask)

        # 1b. Consistency (tokenizer): two recordings of the same word should land on the same
        # pre-quantisation latents, frame by frame along the DTW path, so they get the same codes.
        cons_loss = torch.zeros((), device=x.device)
        if pair_mel is not None and self.consistency_weight > 0:
            has = (pair_index >= 0).any(1)
            if bool(has.any()):
                za = self.vq.pre_norm(self.vq.pre_proj(encoder_out[has]))               # (b, Ta', D)
                zb = self.vq.pre_norm(self.vq.pre_proj(self.encoder(pair_mel[has], pair_padding_mask[has])))
                Ta = za.shape[1]
                pi = pair_index[has][:, ::2][:, :Ta]                                    # latent frame -> pair mel frame
                valid = (pi >= 0) & ~padding_mask[has][:, ::2][:, :Ta]
                j = (pi.clamp(min=0) // 2).clamp(max=zb.shape[1] - 1)                   # pair mel frame -> latent frame
                zb_aligned = zb.gather(1, j.unsqueeze(-1).expand(-1, -1, zb.shape[-1]))
                d = (za - zb_aligned).pow(2).mean(-1)
                cons_loss = (d * valid).sum() / valid.sum().clamp(min=1)

        # 2. Quantize, or (recognizer) predict the codes and look them up in the frozen codebook
        code_logits = None
        if self.predict_codes:
            head_in = encoder_out
            if len(self.code_trunk) > 0:
                # encoder output is at half rate; keep padded latent frames at zero like the encoder does
                lat_pad = padding_mask[:, ::2][:, :encoder_out.shape[1]] if padding_mask is not None else None
                for layer in self.code_trunk:
                    head_in = layer(head_in)
                    if lat_pad is not None:
                        head_in = head_in.masked_fill(lat_pad.unsqueeze(-1), 0.0)
                head_in = self.code_trunk_norm(head_in)
            head_in = self.head_dropout(head_in)
            code_logits = torch.stack([head(head_in) for head in self.code_heads], dim=2)        # (B,T',G,K)
            dec_scores = code_logits
            if not self.training and self.decode in ("smooth", "smooth_soft") and self.smooth_width > 0:
                dec_scores = self.smooth_logprobs(code_logits, self.smooth_width)
            pred_codes = dec_scores.argmax(-1)                                                # (B,T',G)
            if self.merge_rep is not None:
                # merged classes: most likely class (summed probability of its codes), then that class's
                # representative code; applied in training too, so rec2mel sees what inference sees
                p = F.softmax(dec_scores, dim=-1)
                G_ = p.shape[2]
                cls_pred = torch.stack([(p[..., g, :] @ self.merge_onehot(g, p.dtype)).argmax(-1) for g in range(G_)], -1)
                pred_codes = torch.stack([self.merge_rep[g][cls_pred[..., g]] for g in range(G_)], -1)
            if not self.training and self.decode in ("soft", "smooth_soft"):
                quantized = self.probs_to_quantized(F.softmax(dec_scores, dim=-1))
            else:
                quantized = self.codes_to_quantized(pred_codes)
            _, perplexity, active_ratio = self.vq.compute_metrics(
                [F.one_hot(pred_codes[..., g].reshape(-1), self.vq.num_codes).float()
                 for g in range(pred_codes.shape[-1])])
            vq_op = {"quantized": quantized, "codes": pred_codes, "total_loss": torch.zeros((), device=x.device),
                     "perplexity": perplexity, "active_ratio": active_ratio}
            speaker_emb = self.reference_emb.expand(x.shape[0], -1)    # speak in CF02's voice
        else:
            vq_op = self.vq(encoder_out)
            quantized = vq_op["quantized"]
        vq_loss = vq_op["total_loss"]

        # 3. Skip connection (bypasses VQ)
        skip = self.skip_proj(encoder_out)
        if not self.use_skip:                    # codes-only decoding
            skip = torch.zeros_like(skip)

        # 4. Speaker adversary (on VQ output only, not skip)
        if self.mode in ("codes2mel", "recognizer") or self.predict_codes:   # frozen codebook: nothing to shape
            adv_loss = torch.zeros((), device=x.device)
        else:
            adv_loss = self.speaker_adversary(quantized, speaker_ids)

        # 5. Fuse VQ + skip + speaker
        vq_plus_skip = torch.cat([quantized, skip], dim=-1)
        conditioned = self.speaker_injector(vq_plus_skip, speaker_emb)

        # 6. Decode
        predicted = self.decoder(conditioned).transpose(1, 2)

        # 7. Vocoder (optional)
        predicted_waveform = None
        if run_vocoder:
            predicted_waveform = self.mel_to_waveform(predicted)

        # 8. Loss
        mel_loss = torch.tensor(0.0, device=x.device)
        if target is not None:
            min_T = min(predicted.shape[-1], target.shape[-1])
            err = self.freq_weights * (predicted[:, :, :min_T] - target[:, :, :min_T]).abs()   # (B,80,T)
            if padding_mask is not None:
                valid = (~padding_mask[:, :min_T]).unsqueeze(1).to(err.dtype)                   # (B,1,T)
                mel_loss = (err * valid).sum() / (valid.sum() * err.shape[1]).clamp(min=1)
            else:
                mel_loss = err.mean()

        total_loss = mel_loss + vq_loss + self.speaker_adv_weight * adv_loss
        if self.mode in ("codes2mel", "rec2mel"):  # vq / adversary have no trainable parameters here
            total_loss = mel_loss
        if self.consistency_weight > 0:
            total_loss = total_loss + self.consistency_weight * cons_loss

        # 8b. Recognizer: cross-entropy against CF02's codes; mel_loss above is only a monitor
        ce_loss = torch.zeros((), device=x.device)
        code_acc = torch.zeros((), device=x.device)
        code_acc_near = torch.zeros((), device=x.device)
        if code_logits is not None and target_codes is not None:
            Tc = min(code_logits.shape[1], target_codes.shape[1])
            logits = code_logits[:, :Tc]
            tc = target_codes[:, :Tc].to(logits.device)
            tc_multi = None
            if tc.dim() == 4:                    # (B,T',G,R): several reference takes per frame (multi_ref)
                tc_multi, tc = tc, tc[..., 0]    # ref 0 = CF02's own take; accuracies are reported against it
            valid = tc != -100
            tcc = tc.clamp(min=0)
            ce_terms, near_hits, acc_hits = [], [], []
            for g, q in enumerate(self.vq.quantizers):
                with torch.no_grad():
                    E = q.embedding                                                    # (K, D), frozen
                    sq = E.pow(2).sum(1)
                    d = (sq[:, None] - 2 * E @ E.t() + sq[None]).clamp(min=0)         # code-to-code distances
                    near = d.topk(5, dim=1, largest=False).indices                    # 5 nearest (incl. itself)
                logp = F.log_softmax(logits[..., g, :], dim=-1)                       # (B,T',K)
                merged = self.merge_rep is not None
                with torch.no_grad():
                    if tc_multi is not None:
                        # Average of the references' (soft) targets: codes most references agree on dominate,
                        # codes only one take used are diluted.
                        rc = tc_multi[..., g, :]                                       # (B,T',R)
                        rv = (rc != -100).unsqueeze(-1).to(logp.dtype)                 # (B,T',R,1)
                        rc = rc.clamp(min=0)
                        if self.soft_tau > 0:
                            nn_d = d + torch.diag(torch.full_like(sq, float("inf")))
                            tau = self.soft_tau * nn_d.min(1).values.mean().clamp(min=1e-6)
                            per_ref = F.softmax(-d[rc] / tau, dim=-1)                   # (B,T',R,K)
                        else:
                            per_ref = F.one_hot(rc, logp.shape[-1]).to(logp.dtype)
                        soft = (per_ref * rv).sum(-2) / rv.sum(-2).clamp(min=1)       # (B,T',K)
                    elif self.soft_tau > 0:
                        # Soft target: a code acoustically close to CF02's counts as nearly right.
                        # tau scales with the typical nearest-neighbour distance in this group's codebook.
                        nn_d = d + torch.diag(torch.full_like(sq, float("inf")))
                        tau = self.soft_tau * nn_d.min(1).values.mean().clamp(min=1e-6)
                        soft = F.softmax(-d[tcc[..., g]] / tau, dim=-1)               # (B,T',K)
                    else:
                        soft = F.one_hot(tcc[..., g], logp.shape[-1]).to(logp.dtype)
                    if merged:
                        M = self.merge_onehot(g, logp.dtype)                          # (K, classes)
                        soft = soft @ M                                               # target over classes
                if merged:
                    probs_c = logp.exp() @ M                                          # (B,T',classes)
                    loss_g = -(soft * probs_c.clamp(min=1e-12).log()).sum(-1)
                else:
                    loss_g = -(soft * logp).sum(-1)
                vg = valid[..., g]
                ce_terms.append((loss_g * vg).sum() / vg.sum().clamp(min=1))
                pred_g = logits[..., g, :].argmax(-1)
                if merged:   # accuracy over classes: predicted class vs CF02's code's class
                    hit = probs_c.argmax(-1) == self.merge_cls[g][tcc[..., g]]
                else:
                    hit = pred_g == tcc[..., g]
                acc_hits.append((hit & vg).sum().float() / vg.sum().clamp(min=1))
                near_hits.append(((near[tcc[..., g]] == pred_g.unsqueeze(-1)).any(-1) & vg).sum().float()
                                 / vg.sum().clamp(min=1))
            ce_loss = torch.stack(ce_terms).mean()
            code_acc = torch.stack(acc_hits).mean()            # class accuracy when merge_classes is set
            code_acc_near = torch.stack(near_hits).mean()      # predicted code among the target's 5 nearest
        if self.mode == "recognizer":
            if target is not None and target_codes is None:
                raise RuntimeError("recognizer mode needs target_codes: run precompute_target_codes(...) first")
            total_loss = ce_loss

        # 9. vocoder_adapt: core is frozen, so train HiFi-GAN on mel(vocoder(predicted)) vs target.
        # Calls the generator directly: decode_batch/mel_to_waveform run under no_grad.
        if self.mode == "vocoder_adapt" and target is not None:
            wav = self.vocoder.mods.generator(predicted.detach()).squeeze(1)
            mel_fn = self._wav_to_mel[0] = self._wav_to_mel[0].to(wav.device)
            re_mel = torch.log(torch.clamp(mel_fn(wav), min=1e-5))
            min_T = min(re_mel.shape[-1], target.shape[-1])
            # Reported as mel_loss so val/mel_loss (best-ckpt metric) tracks the vocoder.
            mel_loss = (self.freq_weights * (re_mel[:, :, :min_T] - target[:, :, :min_T]).abs()).mean()
            total_loss = mel_loss

        return {
            "predicted": predicted,
            "target": target,
            "loss": total_loss,
            "mel_loss": mel_loss,
            "vq_loss": vq_loss,
            "adv_loss": adv_loss,
            "vq_op": vq_op,
            "predicted_waveform": predicted_waveform,
            "ce_loss": ce_loss,
            "code_acc": code_acc,
            "code_acc_near": code_acc_near,
            "cons_loss": cons_loss,
        }


# Smoke test
if __name__ == "__main__":
    dev = normalize_device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = SpeechReconstructionPipeline(device=dev).to(dev)

    B, T = 32, 45
    model(
        torch.rand(B, N_MELS, T, device=dev),
        torch.randint(0, 2, (B, T), dtype=torch.bool, device=dev),
        torch.randint(0, 100, (B,), device=dev),
        torch.rand(B, SPK_DIM, device=dev),
        torch.rand(B, N_MELS, T, device=dev),
        False,
    )
    print("Forward pass OK")

    model.set_mode("pretrain")

    del model
    if dev.type == "cuda":
        torch.cuda.empty_cache()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Mode: pretrain — 9,027,476 / 45,180,374 params trainable


In [4]:
import os
import time
import argparse
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR
from torch.utils.data import DataLoader, Subset


In [6]:
def load_checkpoint(pipeline, path, device, current_mode, optimizer=None, scheduler=None):

    if Path(path).exists():
        checkpoint = torch.load(path, map_location=device, weights_only=False)

        # Load model weights (always, regardless of mode)
        own = pipeline.state_dict()
        state = {k: v for k, v in checkpoint["model_state_dict"].items()
                 if k not in own or own[k].shape == v.shape}
        skipped = checkpoint["model_state_dict"].keys() - state.keys()
        if skipped:
            print(f"  Skipped shape-mismatched: {sorted(skipped)}")
        missing, unexpected = pipeline.load_state_dict(state, strict=False)
        if missing or unexpected:
            print(f"  WARNING  missing={list(missing)}  unexpected={list(unexpected)}")
        # A codebook whose size changed (e.g. 4x512 -> 2x256) was not loaded, but its 'initialized' flag
        # was: clear it so the new codebook gets the usual data-dependent init on the first batch.
        for name, m in pipeline.named_modules():
            if isinstance(m, VectorQuantizerEMA) and (f"{name}.embedding" in skipped or f"{name}.embedding" in missing):
                m.initialized.zero_()
                m._init_done = False
                print(f"  {name}: new codebook size, will initialise from data")

        saved_mode = checkpoint.get("mode")
        speaker_embs = checkpoint.get("speaker_embeddings")

        # Same mode = resume training (restore everything)
        if saved_mode == current_mode:
            if optimizer and "optimizer_state_dict" in checkpoint:
                optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            if scheduler and "scheduler_state_dict" in checkpoint:
                scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

            best_metric = checkpoint.get("best_metric")
            
            if isinstance(best_metric, dict):
                best_metric = best_metric.get("val/mel_loss", float("inf"))

            print(f"  Loaded checkpoint from {path} (step {checkpoint['step']}, mode: {saved_mode})")
            return (
                checkpoint["step"],
                checkpoint["epoch"],
                best_metric,
                speaker_embs,
            )

        # Different mode = new phase (keep weights, reset everything else)
        print(f"  Loaded weights from {path} (mode: {saved_mode} → {current_mode})")
        print(f"  Resetting step, epoch, metric, optimizer, scheduler")
        return 0, 0, float("inf"), speaker_embs

    else:
        print("No checkpoint found!!")
        return 0, 0, float("inf"), None


def save_checkpoint(pipeline, optimizer, scheduler, step, epoch, path,
                    mode, best_metric=None, speaker_embs=None):

    Path(path).parent.mkdir(parents=True, exist_ok=True)

    torch.save({
        "step": step,
        "epoch": epoch,
        "mode": mode,
        "model_state_dict": pipeline.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_metric": best_metric,
        "speaker_embeddings": speaker_embs,
        "use_skip": pipeline.use_skip,   # read by jasper.load_pipeline; missing = old skip checkpoint
        "predict_codes": pipeline.predict_codes,   # True = recognizer checkpoint (decode via code heads)
    }, path)

In [7]:
def set_checkpoint_mode(path, mode):
    """
    Update the mode stored in a checkpoint file.

    Useful when:
      - Checkpoint was saved without a mode field (older format)
      - You want to force-resume from a different phase's checkpoint
        without resetting optimizer/scheduler state

    Args:
        path: Path to checkpoint file
        mode: "pretrain", "finetune", "e2e", or "vocoder_adapt"
    """
    path = Path(path)
    if not path.exists():
        print(f"Checkpoint not found: {path}")
        return

    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    old_mode = checkpoint.get("mode", "not set")
    checkpoint["mode"] = mode
    torch.save(checkpoint, path)
    print(f"  {path.name}: mode '{old_mode}' → '{mode}'")

set_checkpoint_mode('checkpoints/pretrain_best.pt','pretrain')
set_checkpoint_mode('checkpoints/finetune_best.pt','finetune')
set_checkpoint_mode('checkpoints/e2e_best.pt','e2e')



  pretrain_best.pt: mode 'pretrain' → 'pretrain'
  finetune_best.pt: mode 'finetune' → 'finetune'
  e2e_best.pt: mode 'e2e' → 'e2e'


In [ ]:
import random
from torch.utils.data import Dataset


class Codes2MelDataset(Dataset):
    """Healthy self-reconstruction for codes2mel: LibriTTS + UA-Speech control speakers.

    Control speakers use FINETUNE_BLOCKS only, so the codes2mel gate (CF02's B2 words)
    stays unseen. All mics are kept for channel variety, which makes val slightly
    optimistic (other mics of a take can sit in train); fine for a monitor.
    """

    def __init__(self, max_frames=860, with_pairs=False):
        self.max_frames = max_frames
        self.with_pairs = with_pairs
        self.samples = [(str(p), p.parent.name) for p in sorted(Path(LIBRE_TTS_PATH).rglob("*.pt"))]
        for p in sorted(Path(UA_PATH).rglob("C*.pt")):
            parts = p.stem.split("_")       # CF02_B1_CW75_M8
            if len(parts) == 4 and parts[0] in CONTROL_SPEAKERS and parts[1] in FINETUNE_BLOCKS:
                self.samples.append((str(p), parts[0]))
        self.speakers = {spk for _, spk in self.samples}
        # searched in order for embeddings; control speakers live in their own Kaggle folder
        self.raw_audio_path = [LIBRE_TTS_RAW_PATH, UA_RAW_PATH, UA_CONTROL_RAW_PATH]
        n_ua = sum(spk in CONTROL_SPEAKERS for _, spk in self.samples)
        print(f"  Codes2MelDataset: {len(self.samples) - n_ua} LibriTTS + {n_ua} UA control utterances, "
              f"{len(self.speakers)} speakers")

        # Consistency pairs (tokenizer), UA control utterances only:
        #   same take  = same speaker, block, word, other mic   (channel differs)
        #   same word  = other control speaker or block          (speaker / take differs)
        # Uncommon words (UW*) differ per block, so they only pair within a block.
        self._keys, self._takes, self._words = {}, defaultdict(list), defaultdict(list)
        if with_pairs:
            for i, (path, spk) in enumerate(self.samples):
                parts = Path(path).stem.split("_")
                if spk not in CONTROL_SPEAKERS or len(parts) != 4:
                    continue
                _, blk, word, _mic = parts
                take_key = (spk, blk, word)
                word_key = (blk, word) if word.startswith("UW") else word
                self._keys[i] = (take_key, word_key)
                self._takes[take_key].append(i)
                self._words[word_key].append(i)
            print(f"  consistency pairs: {len(self._keys)} utterances, {len(self._takes)} takes, "
                  f"{len(self._words)} words")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, spk = self.samples[idx]
        mel = torch.load(path, weights_only=False)["mel"]
        if mel.shape[-1] > self.max_frames:
            start = random.randint(0, mel.shape[-1] - self.max_frames)
            mel = mel[:, start:start + self.max_frames]
        T = mel.shape[-1]
        item = {"input_mel": mel, "target_mel": mel.clone(), "speaker_id": spk,
                "input_length": T, "target_length": T}

        if self.with_pairs and idx in self._keys:
            take_key, word_key = self._keys[idx]
            same_take = [j for j in self._takes[take_key] if j != idx]
            other_take = [j for j in self._words[word_key] if self._keys[j][0] != take_key]
            pool = same_take if same_take and (random.random() < 0.5 or not other_take) else other_take
            if pool:
                partner = torch.load(self.samples[random.choice(pool)][0], weights_only=False)["mel"]
                partner = partner[:, :self.max_frames]
                item["pair_mel"] = partner
                item["pair_index"] = dtw_index(mel, partner)
        return item


def dtw_index(a, b, n_mfcc=20):
    """For each frame of mel `a`, the index of its DTW-aligned frame in mel `b`
    (MFCC + CMVN + cosine, as in the dtw_target cell of preprocess.ipynb)."""
    import librosa
    import numpy as np

    def cmvn(f):
        return (f - f.mean(axis=1, keepdims=True)) / (f.std(axis=1, keepdims=True) + 1e-8)

    X = cmvn(librosa.feature.mfcc(S=a.numpy(), n_mfcc=n_mfcc))
    Y = cmvn(librosa.feature.mfcc(S=b.numpy(), n_mfcc=n_mfcc))
    _, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
    idx = np.zeros(a.shape[-1], dtype=np.int64)
    for i, j in wp[::-1]:
        idx[i] = j
    return torch.from_numpy(idx)


In [ ]:


def load_dataloaders(dataset, batch_size, val_fraction=0.05, seed=1234):
    """Build the dataset ONCE and split it.

    The previous version instantiated the dataset twice and returned two loaders
    over the same data, differing only in `shuffle` -- so every "validation"
    metric was a training metric, and the ~136k-file index scan ran twice.

    The split is utterance-level, not speaker-level: holding speakers out would
    leave them absent from `speaker_to_idx`, and `get_speaker_embeddings_for_batch`
    would KeyError on the first val batch.
    """
    base = {"ua-speech": UASpeechDataset, "ua-speech-multi": lambda: UASpeechDataset(multi_ref=True),
            "codes2mel": Codes2MelDataset,
            "tokenizer-pairs": lambda: Codes2MelDataset(with_pairs=True)}.get(dataset, LibreTTSDataset)()

    n = len(base)
   
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed)).tolist()
    n_val = max(batch_size, int(n * val_fraction))
    val_set = Subset(base, perm[:n_val])
    train_set = Subset(base, perm[n_val:])
    
    if hasattr(base, "pairs"):   # UA-Speech: keep all mics of one recording together
        groups = sorted({(p["dysarthric_speaker"], p["word"]) for p in base.pairs})
        perm = torch.randperm(len(groups), generator=torch.Generator().manual_seed(seed)).tolist()
        val_groups = {groups[i] for i in perm[: max(1, int(len(groups) * val_fraction))]}
        key = lambda p: (p["dysarthric_speaker"], p["word"])
        val_idx   = [i for i, p in enumerate(base.pairs) if key(p) in val_groups]
        train_idx = [i for i, p in enumerate(base.pairs) if key(p) not in val_groups]
        val_set, train_set = Subset(base, val_idx), Subset(base, train_idx)

    # Subset hides the attributes the training code reads off .dataset
    for s in (train_set, val_set):
        s.speakers = base.speakers
        s.raw_audio_path = base.raw_audio_path

    common = dict(
        batch_size=batch_size,
        num_workers=4,
        collate_fn=collate_fn,
        pin_memory=True,            # required for .to(non_blocking=True) to overlap
        persistent_workers=True,
    )
    print(f"  {len(train_set)} train / {len(val_set)} val samples")

    return (
        DataLoader(train_set, shuffle=True, drop_last=True, **common),
        DataLoader(val_set, shuffle=False, drop_last=False, **common),
    )
    
train, test = load_dataloaders('ua-speech',16)

print(len(train))

  59166 train / 3113 val samples
3697


In [ ]:
@torch.no_grad()
def precompute_speaker_embeddings(pipeline,speaker_to_idx,raw_audio_dir,device):
   
    # raw_audio_dir may be one folder or a list (codes2mel: LibriTTS + UA-Speech)
    roots = [Path(r) for r in (raw_audio_dir if isinstance(raw_audio_dir, (list, tuple)) else [raw_audio_dir])]
    speaker_embs = {}
    for sid in sorted(speaker_to_idx.keys()):
        wav_files = []
        for root in roots:                       # a folder named after the speaker...
            if (root / sid).exists():
                wav_files = list((root / sid).rglob("*.wav"))
            if wav_files:
                break
        if not wav_files:                        # ...or files named SPK_*.wav anywhere below
            for root in roots:
                wav_files = list(root.rglob(f"{sid}_*.wav"))
                if wav_files:
                    break
            
        if not wav_files:
            print(f"  WARNING: No audio found for speaker {sid}, skipping")
            continue
        
        
        durations = [(sf.info(str(wf)).duration, wf) for wf in wav_files[:50]]
        top_files = [wf for _, wf in sorted(durations, reverse=True)[:10]]
        
        best_file, best_duration = wav_files[0], 0.0
        
        embs =[]
        for wf in top_files:
            waveform, sr = torchaudio.load(str(wf))
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
                
            embs.append(pipeline.extract_speaker_embedding(waveform.to(device), sample_rate=sr).squeeze(0))
            
        
       
        speaker_embs[sid] = F.normalize(torch.stack(embs).mean(dim=0), p=2, dim=-1)

    return speaker_embs

In [10]:
print(LIBRE_TTS_PATH,Path.cwd())

../data/processed/libreTTS/ /teamspace/studios/this_studio/mamba-dsr


In [11]:
def load_speaker_embeddings(dataloader):
  
    
    return {sid:idx for idx, sid in enumerate(sorted(dataloader.dataset.speakers))}

def get_speaker_ids_for_batch(batch, speaker_to_idx, device):
    ids = [speaker_to_idx[sid] for sid in batch['speaker_ids']]
  
    return torch.tensor(ids, dtype=torch.long, device=device)

def get_speaker_embeddings_for_batch(batch, speaker_embs, device):
    embs = torch.stack([speaker_embs[sid] for sid in batch['speaker_ids']])
    return embs.to(device)










In [12]:
import random


def spec_augment(mel, padding_mask, n_freq=2, max_freq=10, n_time=2, max_time=20):
    """SpecAugment for (B, 80, T) log-mels: random frequency bands and time spans are set to the
    utterance's mean value. Training only (recognizer); the target is never masked."""
    mel = mel.clone()
    valid = (~padding_mask).unsqueeze(1).to(mel.dtype)                                # (B,1,T)
    fill = (mel * valid).sum((1, 2)) / (valid.sum((1, 2)) * mel.shape[1]).clamp(min=1)
    n_bins = mel.shape[1]
    for b, L in enumerate((~padding_mask).sum(1).tolist()):
        for _ in range(n_freq):
            w = random.randint(0, max_freq)
            f0 = random.randint(0, n_bins - w)
            mel[b, f0:f0 + w, :L] = fill[b]
        for _ in range(n_time):
            w = random.randint(0, min(max_time, max(1, L // 5)))
            t0 = random.randint(0, max(0, L - w))
            mel[b, :, t0:t0 + w] = fill[b]
    return mel


def train_step(pipeline,batch, optimizer, scheduler,
               speaker_embs, speaker_to_idx, device, step,config):

    input_mel = batch['input_mel'].to(device, non_blocking=True)
    target_mel = batch['target_mel'].to(device, non_blocking=True)
    padding_mask = batch['padding_mask'].to(device, non_blocking=True)


    speaker_emb = get_speaker_embeddings_for_batch(batch,speaker_embs,device)
    speaker_ids = get_speaker_ids_for_batch(batch, speaker_to_idx, device)



    pipeline.set_adversary_scale(step)

    # run_vocoder=False: the waveform is not part of the loss.
    target_codes = batch.get("target_codes")
    if target_codes is not None:
        target_codes = target_codes.to(device, non_blocking=True)
    if config.get("specaug", False):
        input_mel = spec_augment(input_mel, padding_mask)
    pair = {k: batch[k].to(device, non_blocking=True)
            for k in ("pair_mel", "pair_padding_mask", "pair_index") if k in batch}
    result = pipeline(
        input_mel,padding_mask,speaker_ids,speaker_emb,target_mel, False, target_codes=target_codes, **pair
    )

    result["loss"].backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in pipeline.parameters() if p.requires_grad],
        max_norm=config["grad_clip"],
    )

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    # float() not .item(): drops the autograd graph so logged metrics can't pin it
    vq_op = result["vq_op"]
    metrics = {
        "train/mel_loss": float(result["mel_loss"]),
        "train/vq_loss": float(result["vq_loss"]),
        "train/adv_loss": float(result["adv_loss"]),
        "train/total_loss": float(result["loss"]),
        "train/perplexity": float(vq_op["perplexity"]),
        "train/active_ratio": float(vq_op["active_ratio"]),
        "train/grad_norm": float(grad_norm),
        "train/lr": scheduler.get_last_lr()[0],
    }
    if pipeline.predict_codes:
        metrics["train/ce_loss"] = float(result["ce_loss"])
        metrics["train/code_acc"] = float(result["code_acc"])
        metrics["train/code_acc_near"] = float(result["code_acc_near"])
    if pipeline.consistency_weight > 0:
        metrics["train/cons_loss"] = float(result["cons_loss"])
    return metrics




In [ ]:
def codebook_arch(state):
    """Sizes stored in a state_dict: {'vq_groups', 'vq_codes', 'code_dim', 'head_layers'}."""
    groups = {k.split(".")[2] for k in state if k.startswith("vq.quantizers.") and k.endswith(".embedding")}
    codes, code_dim = state["vq.quantizers.0.embedding"].shape
    head_layers = len({k.split(".")[1] for k in state if k.startswith("code_trunk.")})   # 0 = no extra layers
    merge_classes = state["merge_rep"].shape[1] if "merge_rep" in state else 0         # 0 = raw codes
    return dict(vq_groups=len(groups), vq_codes=codes, code_dim=code_dim, head_layers=head_layers,
                merge_classes=merge_classes)


def load_frozen_tokenizer(tokenizer_checkpoint, device):
    """Pipeline with a tokenizer checkpoint's weights, in eval mode, decoding from its own VQ codes."""
    ckpt = torch.load(tokenizer_checkpoint, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"]
    adv = [k for k in state if k.startswith("speaker_adversary") and k.endswith("weight")]
    tok = SpeechReconstructionPipeline(
        n_mels=N_MELS, d_model=D_MODEL, n_layers=N_LAYERS, **codebook_arch(state),
        spk_dim=SPK_DIM, num_speakers=state[adv[-1]].shape[0], device=device,
    ).to(device)
    tok.load_state_dict(state, strict=False)
    tok.use_skip, tok.predict_codes = False, False
    return tok.eval()                           # no EMA / dead-code resets: the codebook must not move


def code_merge_map(checkpoint, n_classes=128, seed=0):
    """Merge each group's codes into n_classes classes: k-means on the code embeddings, weighted by
    how much each code is used (EMA counts), so used codes anchor the classes and near-duplicate or
    rarely used codes join their nearest one. The tokenizer's dead-code resets leave many near-twin
    codes that split one sound into several indices; merging them makes the labels consistent.

    Returns {"n_classes", "cls": (G, K) code -> class, "rep": (G, n_classes) the most-used code of
    each class (what the decoder is fed for that class)}.
    """
    from sklearn.cluster import KMeans
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)["model_state_dict"]
    G = codebook_arch(state)["vq_groups"]
    cls, reps = [], []
    for g in range(G):
        E = state[f"vq.quantizers.{g}.embedding"].float()
        w = state[f"vq.quantizers.{g}.ema_cluster_size"].float()
        w = (w / w.sum()).numpy() + 1e-6
        km = KMeans(n_clusters=n_classes, n_init=10, random_state=seed).fit(E.numpy(), sample_weight=w)
        lab = torch.as_tensor(km.labels_, dtype=torch.long)
        r = torch.zeros(n_classes, dtype=torch.long)
        for c in range(n_classes):
            members = (lab == c).nonzero().squeeze(1)
            r[c] = members[torch.as_tensor(w)[members].argmax()] if len(members) else 0
        cls.append(lab); reps.append(r)
    return {"n_classes": n_classes, "cls": torch.stack(cls), "rep": torch.stack(reps)}


@torch.no_grad()
def code_agreement_check(tokenizer_checkpoint, speaker=REFERENCE_CONTROL, blocks=("B1", "B3"), mic="M5",
                         device="cuda:0", root=UA_PATH, merge_classes=(None,)):
    """How predictable are the tokenizer's codes at all? Takes `speaker`'s two recordings of the
    same word from two blocks (shared words only: digits, letters, commands, common words), DTW-aligns
    them (same MFCC + CMVN + cosine as the dtw_target cell) and reports how often the two clean takes
    get the same code (exact), a code among each other's 5 nearest (near, unmerged only), the agreement
    expected by chance, and exact agreement above chance = (exact - chance) / (1 - chance).

    merge_classes: codebook sizes to evaluate, None = the raw codes, an int = codes merged into that
    many classes per group with code_merge_map, e.g. (None, 256, 128, 64). Compare with the
    recognizer's val code_acc / code_acc_near.
    """
    import librosa
    import numpy as np
    device = normalize_device(device)
    tok = load_frozen_tokenizer(tokenizer_checkpoint, device)

    takes = {}
    for p in sorted(Path(root).glob(f"{speaker}/{speaker}_*_{mic}.pt")):
        parts = p.stem.split("_")
        if len(parts) != 4 or parts[2].startswith("UW"):    # uncommon words differ per block
            continue
        takes.setdefault(parts[2], {})[parts[1]] = p
    pairs = [(d[blocks[0]], d[blocks[1]]) for d in takes.values() if blocks[0] in d and blocks[1] in d]

    def cmvn(f):
        return (f - f.mean(axis=1, keepdims=True)) / (f.std(axis=1, keepdims=True) + 1e-8)

    near = []
    for q in tok.vq.quantizers:
        Eq = q.embedding
        sq = Eq.pow(2).sum(1)
        near.append((sq[:, None] - 2 * Eq @ Eq.t() + sq[None]).topk(5, dim=1, largest=False).indices)

    G = len(tok.vq.quantizers)
    As, Bs = [], []
    for pa, pb in pairs:
        ma = torch.load(pa, weights_only=False)["mel"]
        mb = torch.load(pb, weights_only=False)["mel"]
        ca = tok.vq(tok.encoder(ma[None].to(device), None))["codes"][0]     # (Ta', G)
        cb = tok.vq(tok.encoder(mb[None].to(device), None))["codes"][0]
        X = cmvn(librosa.feature.mfcc(S=ma.numpy(), n_mfcc=20))
        Y = cmvn(librosa.feature.mfcc(S=mb.numpy(), n_mfcc=20))
        _, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
        path = np.unique(np.asarray(wp) // 2, axis=0)                       # mel frames -> code frames
        path = path[(path[:, 0] < ca.shape[0]) & (path[:, 1] < cb.shape[0])]
        As.append(ca[torch.as_tensor(path[:, 0], device=device)])
        Bs.append(cb[torch.as_tensor(path[:, 1], device=device)])
    A, B = torch.cat(As), torch.cat(Bs)                                      # (n, G) aligned code pairs
    n = A.shape[0]
    print(f"{speaker} {blocks[0]} vs {blocks[1]} ({mic}): {len(pairs)} words, {n} aligned code frames")

    for K in merge_classes:
        if K is None:
            a, b, size = A, B, tok.vq.num_codes
        else:
            cls = code_merge_map(tokenizer_checkpoint, K)["cls"].to(device)   # (G, codes)
            a = torch.stack([cls[g][A[:, g]] for g in range(G)], 1)
            b = torch.stack([cls[g][B[:, g]] for g in range(G)], 1)
            size = K
        ex, ch = [], []
        for g in range(G):
            u = torch.bincount(a[:, g], minlength=size) + torch.bincount(b[:, g], minlength=size)
            ch.append(float(((u / u.sum()) ** 2).sum()))
            ex.append(float((a[:, g] == b[:, g]).float().mean()))
        ex, ch = np.array(ex), np.array(ch)
        above = (ex - ch) / (1 - ch)
        line = (f"  {'raw ' + str(size) if K is None else 'merged ' + str(K):>11} codes: exact {100 * ex.mean():5.1f}%   "
                f"chance {100 * ch.mean():4.1f}%   above chance {100 * above.mean():5.1f}%")
        if K is None:
            close = np.mean([float((near[g][A[:, g]] == B[:, g:g + 1]).any(-1).float().mean()) for g in range(G)])
            line += f"   near-5 {100 * close:5.1f}%"
        print(line)
    del tok
    torch.cuda.empty_cache()


@torch.no_grad()
def precompute_target_codes(tokenizer_checkpoint, device="cuda:0", root=UA_PATH):
    """Recognizer targets: encode each dysarthric file's dtw_target (CF02's take of the word,
    DTW-warped to the dysarthric timing) with the FROZEN tokenizer, and store the code indices
    (T', groups) in the same .pt as "target_codes". Run once per tokenizer; files already
    encoded with this tokenizer are skipped. Covers all blocks (B2 is never trained on).

    Note: this edits the local processed/ua files; re-zip to Drive or rerun in a new session.
    """
    from tqdm.auto import tqdm
    device = normalize_device(device)
    tok = load_frozen_tokenizer(tokenizer_checkpoint, device)

    tag = Path(tokenizer_checkpoint).name
    stats = {"encoded": 0, "already": 0, "no_dtw_target": 0}
    for f in tqdm(sorted(Path(root).glob("[FM]*/*.pt")), desc="target codes"):   # dysarthric dirs only
        d = torch.load(f, weights_only=False)
        if "dtw_target" not in d:
            stats["no_dtw_target"] += 1
            continue
        if d.get("target_codes_tokenizer") == tag:
            stats["already"] += 1
            continue
        codes = tok.vq(tok.encoder(d["dtw_target"].unsqueeze(0).to(device), None))["codes"][0]   # (T', G)
        d["target_codes"] = codes.to(torch.int16).cpu()
        d["target_codes_tokenizer"] = tag
        torch.save(d, f)
        stats["encoded"] += 1
    print(stats)


@torch.no_grad()
def precompute_multi_target_codes(tokenizer_checkpoint, refs=("CF02", "CF03", "CF04", "CF05"), max_refs=8,
                                  ref_mic="M5", device="cuda:0", root=UA_PATH):
    """Multi-reference recognizer targets. For every dysarthric B1/B3 file, take other healthy takes of
    the same word -- CF02's take from the other block (shared words only) and the `refs` speakers' takes
    from either training block -- DTW-warp each to the dysarthric timing (same MFCC + CMVN + cosine
    as dtw_target) and encode it with the FROZEN tokenizer. Stored as "target_codes_multi"
    (T', groups, max_refs) int16: ref 0 = the existing target_codes (CF02, same take), the rest the
    extra references, -100 where there are fewer. Run precompute_target_codes with the same
    tokenizer first. B2 is never touched, so the gate and Jasper sets stay unseen.
    """
    import librosa
    import numpy as np
    from tqdm.auto import tqdm
    device = normalize_device(device)
    tok = load_frozen_tokenizer(tokenizer_checkpoint, device)
    tag = Path(tokenizer_checkpoint).name
    multi_tag = f"{tag}|{','.join(refs)}|{max_refs}|{ref_mic}"

    def feats(mel):
        f = librosa.feature.mfcc(S=mel.numpy(), n_mfcc=20)
        return (f - f.mean(axis=1, keepdims=True)) / (f.std(axis=1, keepdims=True) + 1e-8)

    # one recording per reference take (speaker, block, word): ref_mic if present, else the first mic
    take_file = {}
    for spk in refs:
        for p in sorted(Path(root).rglob(f"{spk}_*.pt")):
            parts = p.stem.split("_")
            if len(parts) != 4 or parts[1] not in FINETUNE_BLOCKS:
                continue
            key = (spk, parts[1], parts[2])
            if key not in take_file or parts[3] == ref_mic:
                take_file[key] = p
    by_word = defaultdict(list)            # uncommon words (UW*) differ per block, the rest are shared
    for (spk, blk, word), p in sorted(take_file.items()):
        by_word[(blk, word) if word.startswith("UW") else word].append((spk, blk, p))
    print(f"  {len(take_file)} reference takes from {', '.join(refs)} ({len(by_word)} words)")

    cache = {}
    def ref(p):
        if p not in cache:
            m = torch.load(p, weights_only=False)["mel"]
            cache[p] = (m, feats(m))
        return cache[p]

    stats = {"encoded": 0, "already": 0, "no_target_codes": 0, "refs_total": 0}
    files = [f for f in sorted(Path(root).glob("[FM]*/*.pt")) if f.stem.split("_")[1] in FINETUNE_BLOCKS]
    for f in tqdm(files, desc="multi-ref codes"):
        d = torch.load(f, weights_only=False)
        if d.get("target_codes_tokenizer") != tag:
            stats["no_target_codes"] += 1        # run precompute_target_codes(tokenizer_checkpoint) first
            continue
        if d.get("target_codes_multi_tag") == multi_tag:
            stats["already"] += 1
            continue
        _, blk, word, _ = f.stem.split("_")
        others = [p for spk, b, p in by_word.get((blk, word) if word.startswith("UW") else word, [])
                  if not (spk == REFERENCE_CONTROL and b == blk)][: max_refs - 1]   # CF02 same take = ref 0
        base = torch.as_tensor(d["target_codes"]).long()                          # (T', G)
        multi = torch.full((*base.shape, max_refs), -100, dtype=torch.int16)
        multi[..., 0] = base.to(torch.int16)
        if others:
            mel = d["mel"]
            X = feats(mel)
            warped = []
            for p in others:
                rm, Y = ref(p)
                _, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
                idx = np.zeros(mel.shape[-1], dtype=np.int64)
                for i, j in wp[::-1]:
                    idx[i] = j
                warped.append(rm[:, torch.from_numpy(idx)])
            codes = tok.vq(tok.encoder(torch.stack(warped).to(device), None))["codes"]   # (R', T', G)
            n = min(codes.shape[1], base.shape[0])
            multi[:n, :, 1:1 + len(others)] = codes[:, :n].permute(1, 2, 0).to(torch.int16).cpu()
            stats["refs_total"] += len(others)
        d["target_codes_multi"] = multi
        d["target_codes_multi_tag"] = multi_tag
        torch.save(d, f)
        stats["encoded"] += 1
    stats["mean_extra_refs"] = round(stats["refs_total"] / max(1, stats["encoded"]), 2)
    print(stats)

In [13]:
import soundfile as sf

@torch.no_grad()
def validate(pipeline, val_loader, speaker_embs, speaker_to_idx, device, max_batches=50):
    """Returns the metrics dict the training loop expects (needs 'val/mel_loss').

    eval() matters beyond dropout here: it stops the VQ's EMA updates and dead-code
    resets from firing on validation batches and mutating the codebook.
    """
    pipeline.eval()
    totals, n = defaultdict(float), 0

    for i, batch in enumerate(val_loader):
        if i >= max_batches:
            break

        input_mel    = batch['input_mel'].to(device, non_blocking=True)
        target_mel   = batch['target_mel'].to(device, non_blocking=True)
        padding_mask = batch['padding_mask'].to(device, non_blocking=True)

        speaker_emb = get_speaker_embeddings_for_batch(batch, speaker_embs, device)
        speaker_ids = get_speaker_ids_for_batch(batch, speaker_to_idx, device)

        target_codes = batch.get("target_codes")
        if target_codes is not None:
            target_codes = target_codes.to(device, non_blocking=True)
        pair = {k: batch[k].to(device, non_blocking=True)
                for k in ("pair_mel", "pair_padding_mask", "pair_index") if k in batch}
        result = pipeline(input_mel, padding_mask, speaker_ids,
                          speaker_emb, target_mel, False, target_codes=target_codes, **pair)

        totals["val/mel_loss"]   += float(result["mel_loss"])
        totals["val/vq_loss"]    += float(result["vq_loss"])
        totals["val/adv_loss"]   += float(result["adv_loss"])
        totals["val/total_loss"] += float(result["loss"])
        totals["val/perplexity"] += float(result["vq_op"]["perplexity"])
        if pipeline.predict_codes:
            totals["val/ce_loss"]  += float(result["ce_loss"])
            totals["val/code_acc"] += float(result["code_acc"])
            totals["val/code_acc_near"] += float(result["code_acc_near"])
        if pipeline.consistency_weight > 0:
            totals["val/cons_loss"] += float(result["cons_loss"])
        n += 1

    return {k: v / max(1, n) for k, v in totals.items()}


def train(mode, checkpoint_path, device):

    device = normalize_device(device)

    config = MODE_CONFIG[mode]

    dataset = {"pretrain": "libre-tts", "codes2mel": "codes2mel",
               "tokenizer": "tokenizer-pairs" if config.get("consistency_weight", 0) > 0 else "codes2mel",
               "recognizer": "ua-speech-multi" if config.get("multi_ref", False) else "ua-speech",
               }.get(mode, "ua-speech")
    print(f"  dataset: {dataset}")
    train_loader, val_loader = load_dataloaders(dataset, config['batch_size'])


    output_dir = "./checkpoints"



    start_step = 0
    start_epoch = 0
    best_metric = float("inf")
    speaker_embs = None
    speaker_to_idx = load_speaker_embeddings(train_loader)
    print(f"{len(speaker_to_idx)} Speakers")

    # New codebooks use the constants; modes that keep a frozen codebook take its sizes from the checkpoint.
    arch = dict(vq_groups=VQ_GROUPS, vq_codes=VQ_CODES, code_dim=CODE_DIM)
    if mode in ("recognizer", "codes2mel", "rec2mel") and checkpoint_path and Path(checkpoint_path).exists():
        arch = codebook_arch(torch.load(checkpoint_path, map_location="cpu", weights_only=False)["model_state_dict"])
    if mode == "recognizer":   # the tokenizer checkpoint has no trunk; its size comes from the config
        arch["head_layers"] = config.get("head_layers", arch.get("head_layers", 0))
        arch["merge_classes"] = config.get("merge_classes") or 0
    elif mode != "rec2mel":    # rec2mel keeps the recognizer checkpoint's trunk size and merge map
        arch.pop("head_layers", None)
        arch.pop("merge_classes", None)
    print(f"  codebook: {arch['vq_groups']} groups x {arch['vq_codes']} codes x {arch['code_dim']} dims")

    pipeline = SpeechReconstructionPipeline(

            n_mels = N_MELS,
            d_model = D_MODEL,
            n_layers = N_LAYERS,
            **arch,
            spk_dim = SPK_DIM,
            num_speakers = len(speaker_to_idx),
            speaker_adv_weight = config.get("adv_weight", 0.1),
            device = device,

        )

    pipeline = pipeline.to(device)
    pipeline.set_mode(mode=mode)
    pipeline.consistency_weight = config.get("consistency_weight", 0.0)
    if pipeline.consistency_weight > 0:
        print(f"  consistency loss on, weight {pipeline.consistency_weight}")

    if mode == "recognizer":
        pipeline.soft_tau = config.get("soft_tau", 0.0)
        pipeline.head_dropout.p = config.get("head_dropout", 0.0)
        if config.get("freeze_lower_encoder", False):
            enc = pipeline.encoder
            for m in [enc.input_proj, enc.downsample, *enc.layers[:enc.downsample_after]]:
                for p in m.parameters():
                    p.requires_grad = False
        n_train = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
        print(f"  recognizer: head_layers={len(pipeline.code_trunk)}, soft_tau={pipeline.soft_tau}, "
              f"head_dropout={pipeline.head_dropout.p}, "
              f"freeze_lower_encoder={config.get('freeze_lower_encoder', False)}, "
              f"specaug={config.get('specaug', False)} -> {n_train:,} params trainable")

    

    if checkpoint_path:
            start_step, start_epoch ,saved_metric, speaker_embs = load_checkpoint(pipeline, checkpoint_path, device=device,current_mode=mode)
            if saved_metric is not None:
                best_metric = saved_metric
            # Recognizer: the decoder speaks in REFERENCE_CONTROL's voice. Its embedding comes from the
            # tokenizer checkpoint (UA control speakers); on resume it is restored from reference_emb.
            if mode == "recognizer" and speaker_embs is not None and REFERENCE_CONTROL in speaker_embs:
                pipeline.reference_emb.copy_(speaker_embs[REFERENCE_CONTROL].to(pipeline.reference_emb.device))
                print(f"  Decoder voice: {REFERENCE_CONTROL} embedding from {checkpoint_path}")
            # Checkpoint from another dataset (pretrain -> finetune): its speaker embs, step
            # and val loss don't apply here, so start the stage fresh with only the weights.
            if speaker_embs is not None and set(speaker_embs) != set(speaker_to_idx):
                # Checkpoint from another dataset: its step and val loss don't apply here. Embeddings of
                # speakers in both are reused (pretrain -> codes2mel keeps LibriTTS's 246, adds 13 controls).
                print("  Checkpoint is from a different dataset: resetting step/metric, reusing shared speaker embs")
                start_step, start_epoch, best_metric = 0, 0, float("inf")
                speaker_embs = {s: v for s, v in speaker_embs.items() if s in speaker_to_idx}

    if mode == "recognizer" and float(pipeline.reference_emb.abs().sum()) == 0:
        raise RuntimeError(f"recognizer needs a tokenizer checkpoint with a {REFERENCE_CONTROL} embedding")
    if mode == "recognizer" and pipeline.merge_rep is not None:
        mm = code_merge_map(checkpoint_path, pipeline.merge_rep.shape[1])
        pipeline.merge_cls.copy_(mm["cls"].to(pipeline.merge_cls.device))
        pipeline.merge_rep.copy_(mm["rep"].to(pipeline.merge_rep.device))
        print(f"  merged codes: {pipeline.merge_rep.shape[1]} classes per group (code_acc = class accuracy)")
    if mode == "rec2mel" and pipeline.merge_rep is not None:
        print(f"  rec2mel: recognizer predicts {pipeline.merge_rep.shape[1]} merged classes per group")
    if mode == "rec2mel":
        if float(pipeline.reference_emb.abs().sum()) == 0 or float(pipeline.code_heads[0].weight.abs().sum()) == 0:
            raise RuntimeError("rec2mel needs a recognizer checkpoint (code heads + reference_emb)")
        print(f"  rec2mel: recognizer frozen ({len(pipeline.code_trunk)} trunk layers), "
              f"training injector + decoder on its argmax codes")

    missing = sorted(set(speaker_to_idx) - set(speaker_embs or {}))
    if missing:
        print(f"  Computing speaker embeddings for {len(missing)} speakers")
        speaker_embs = {**(speaker_embs or {}),
                        **precompute_speaker_embeddings(pipeline, {s: 0 for s in missing},
                                                        train_loader.dataset.raw_audio_path, device)}
        still_missing = sorted(set(speaker_to_idx) - set(speaker_embs))
        if still_missing:
            raise RuntimeError(f"No raw audio found for speakers {still_missing}; check raw_audio_path")


    trainable_params = [p for p in pipeline.parameters() if p.requires_grad]
    optimizer = AdamW(
        trainable_params,
        lr=config["lr"],
        betas=(0.9, 0.98),
        weight_decay=0.01,
    )

    total_steps = config["epochs"] * len(train_loader)
    warmup_steps = config['warmup_steps']

    

    def lambda_lr(step):
        if(step < warmup_steps):
            return step/max(1,warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1.0 + __import__("math").cos(__import__("math").pi * progress)))

    scheduler = LambdaLR(optimizer=optimizer,lr_lambda=lambda_lr)

    global_step = start_step

    epochs = config['epochs']



    pipeline.train();
    print(f"\nStarting training: {total_steps} steps ({config['epochs']} epochs)")
    print(f"  Batch size: {config['batch_size']}")
    print(f"  Learning rate: {config['lr']}")
    print(f"  Warmup: {config['warmup_steps']} steps")
    print(f"  Grad clip: {config['grad_clip']}")
    print(f"  Start Epoch: {start_epoch}")
    print(f"  Start Step: {global_step}")
    print(f"  Best Metric : {best_metric}")
    


    





    batches_per_epoch = len(train_loader)

    for epoch in range(start_epoch,epochs):

        epoch_start = time.time()

        for batch_idx, batch in enumerate(train_loader):
            if epoch == start_epoch and global_step > start_step:
                pass  # already past this point
            elif epoch == start_epoch and batch_idx < (start_step % batches_per_epoch):
                continue

            global_step+=1

            metrics = train_step(pipeline,batch, optimizer, scheduler,
                       speaker_embs, speaker_to_idx, device, global_step,config)

           

            # --- Validate ---
            if global_step % config["eval_interval"] == 0:
                val_metrics = validate(
                    pipeline, val_loader, speaker_embs, speaker_to_idx, device,
                )

                print(f"  [VAL] step {global_step} | " + " | ".join(
                    f"{k}: {v:.4f}" for k, v in val_metrics.items()
                ))
                print(metrics)
                # Save best checkpoint
                # recognizer: pick checkpoints on code cross-entropy (mel_loss is only a monitor there)
                val_loss = val_metrics["val/ce_loss" if mode == "recognizer" else "val/mel_loss"]
                if val_loss < best_metric:
                    best_metric = val_loss
                    best_path = os.path.join(output_dir, f"{mode}_best.pt")
                    save_checkpoint(
                        pipeline, optimizer, scheduler,
                        global_step, epoch, best_path, mode ,best_metric,
                        speaker_embs,
                    )
                    print(f"  [BEST] New best val loss: {best_metric:.4f} → {best_path}")


                pipeline.train()
            
        epoch_time = time.time() - epoch_start
        print(f"\n  Epoch {epoch + 1}/{config['epochs']} done in {epoch_time:.1f}s")
        print()
    final_path = os.path.join(output_dir, f"{mode}_final.pt")
    save_checkpoint(
        pipeline, optimizer, scheduler,
        global_step, config["epochs"], final_path, mode ,best_metric,
        speaker_embs,
    )







In [ ]:
# train('pretrain','./checkpoint.pt','cuda:0')


<generator object Path.rglob at 0x7b2cd9a8f010>


31106
  29551 train / 1555 val samples
246 Speakers


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Mode: pretrain — 8,914,726 / 45,067,624 params trainable
No checkpoint found!!
  103: 103_1241_000074_000001.wav (4.0s)
  1034: 1034_121119_000052_000002.wav (11.1s)
  1040: 1040_133433_000055_000005.wav (4.8s)
  1069: 1069_133699_000067_000000.wav (9.5s)
  1081: 1081_128618_000024_000003.wav (4.0s)
  1088: 1088_129236_000028_000005.wav (8.2s)
  1098: 1098_133695_000015_000001.wav (16.0s)
  1116: 1116_132847_000004_000001.wav (9.2s)
  118: 118_47824_000107_000000.wav (4.6s)
  1183: 1183_124566_000011_000001.wav (7.1s)
  1235: 1235_135884_000005_000005.wav (12.0s)
  1246: 1246_124550_000008_000000.wav (20.4s)
  125: 125_121342_000043_000013.wav (7.2s)
  1263: 1263_141777_000029_000002.wav (9.6s)
  1334: 1334_135589_000092_000004.wav (18.3s)
  1355: 1355_39947_000000_000009.wav (17.7s)
  1363: 1363_139304_000005_000001.wav (12.1s)
  1447: 1447_17506_000052_000003.wav (15.7s)
  1455: 1455_134435_000015_000003.wav (9.8s)
  150: 150_126112_000002_000005.wav (3.8s)
  1502: 1502_122615_000044

/tmp/ipykernel_129372/3481616918.py:37: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  "train/mel_loss": float(result["mel_loss"]),



  Epoch 1/50 done in 712.2s

  [VAL] step 2000 | val/mel_loss: 0.5136 | val/vq_loss: 1.7456 | val/adv_loss: 5.2526 | val/total_loss: 2.7844 | val/perplexity: 34.6012
{'train/mel_loss': 0.5729638338088989, 'train/vq_loss': 2.991076707839966, 'train/adv_loss': 5.136486053466797, 'train/total_loss': 4.077689170837402, 'train/perplexity': 394.4104919433594, 'train/active_ratio': 0.93408203125, 'train/grad_norm': 2.484259605407715, 'train/lr': 0.00011999999999999999}
  [BEST] New best val loss: 0.5136 → ./checkpoints/pretrain_best.pt


In [ ]:
# train('finetune','./checkpoints/pretrain_best.pt','cuda:0')

62279
  59166 train / 3113 val samples
14 Speakers


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Mode: finetune — 5,802,526 / 45,004,864 params trainable
  Skipped shape-mismatched: ['speaker_adversary.classifier.3.bias', 'speaker_adversary.classifier.3.weight']
  WARNING  missing=['speaker_adversary.classifier.3.weight', 'speaker_adversary.classifier.3.bias']  unexpected=[]
  Loaded weights from ./checkpoints/pretrain_best.pt (mode: pretrain → finetune)
  Resetting step, epoch, metric, optimizer, scheduler
  Checkpoint is from a different dataset: resetting step/metric/speaker embs
  F02: F02_B1_UW46_M2.wav (9.9s)
  F03: F03_B1_UW4_M5.wav (4.6s)
  F04: F04_B2_UW54_M7.wav (4.8s)
  F05: F05_B3_UW71_M7.wav (25.6s)
  M01: M01_B1_C17_M5.wav (16.4s)
  M04: M04_B1_UW18_M6.wav (3.5s)
  M05: M05_B3_UW2_M6.wav (21.7s)
  M07: M07_B3_C10_M5.wav (5.0s)
  M08: M08_B1_UW50_M4.wav (5.2s)
  M09: M09_B1_UW20_M7.wav (4.7s)
  M10: M10_B3_UW84_M4.wav (3.1s)
  M11: M11_B1_LD_M8.wav (4.2s)
  M12: M12_B2_UW30_M8.wav (9.3s)
  M14: M14_B3_UW59_M4.wav (2.8s)

Starting training: 221850 steps (30 epochs)
  B

/tmp/ipykernel_3620/3481616918.py:37: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  "train/mel_loss": float(result["mel_loss"]),


  [VAL] step 500 | val/mel_loss: 1.2408 | val/vq_loss: 2.1002 | val/adv_loss: 2.6517 | val/total_loss: 3.6062 | val/perplexity: 70.6481
{'train/mel_loss': 1.289415955543518, 'train/vq_loss': 2.878087043762207, 'train/adv_loss': 2.5450618267059326, 'train/total_loss': 4.422008991241455, 'train/perplexity': 301.4050598144531, 'train/active_ratio': 0.79638671875, 'train/grad_norm': 1.1427096128463745, 'train/lr': 1e-05}
  [BEST] New best val loss: 1.2408 → ./checkpoints/finetune_best.pt
  [VAL] step 1000 | val/mel_loss: 1.1487 | val/vq_loss: 1.9454 | val/adv_loss: 2.6090 | val/total_loss: 3.3549 | val/perplexity: 52.5900
{'train/mel_loss': 0.7522700428962708, 'train/vq_loss': 2.7427051067352295, 'train/adv_loss': 2.6872353553771973, 'train/total_loss': 3.7636985778808594, 'train/perplexity': 234.8703155517578, 'train/active_ratio': 0.7783203125, 'train/grad_norm': 2.8603928089141846, 'train/lr': 2e-05}
  [BEST] New best val loss: 1.1487 → ./checkpoints/finetune_best.pt
  [VAL] step 1500 |

KeyboardInterrupt: 

In [ ]:
# train('e2e','./checkpoints/finetune_best.pt','cuda:0')

62279
  59166 train / 3113 val samples
14 Speakers


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Mode: e2e — 8,868,414 / 45,004,864 params trainable
  Loaded weights from ./checkpoints/finetune_best.pt (mode: finetune → e2e)
  Resetting step, epoch, metric, optimizer, scheduler

Starting training: 73950 steps (10 epochs)
  Batch size: 8
  Learning rate: 1e-05
  Warmup: 0 steps
  Grad clip: 0.5
  Start Epoch: 0
  Start Step: 0
  Best Metric : inf
  [VAL] step 500 | val/mel_loss: 0.9586 | val/vq_loss: 1.9217 | val/adv_loss: 2.5275 | val/total_loss: 3.1330 | val/perplexity: 50.4561
{'train/mel_loss': 0.6772911548614502, 'train/vq_loss': 2.7920632362365723, 'train/adv_loss': 2.576638698577881, 'train/total_loss': 3.727018356323242, 'train/perplexity': 264.4080810546875, 'train/active_ratio': 0.78759765625, 'train/grad_norm': 5.505947589874268, 'train/lr': 9.998872057198983e-06}
  [BEST] New best val loss: 0.9586 → ./checkpoints/e2e_best.pt
  [VAL] step 1000 | val/mel_loss: 0.9406 | val/vq_loss: 1.8353 | val/adv_loss: 2.5263 | val/total_loss: 3.0285 | val/perplexity: 42.4651
{'train/me

Exception in thread Thread-6 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/

KeyboardInterrupt: 

In [ ]:
# Code-prediction pipeline, in order (gates and evaluation are in jasper.ipynb).
# 1. Tokenizer: the whole core retrained without skip on LibriTTS + UA control speakers.
# train('tokenizer','./checkpoints/pretrain_best.pt','cuda:0')
#    Consistency loss: fine-tune a relabelled copy (a same-mode checkpoint would resume instead).
# !cp checkpoints/tokenizer_best.pt checkpoints/tokenizer_init.pt
# set_checkpoint_mode('checkpoints/tokenizer_init.pt', 'pretrain')
# train('tokenizer','./checkpoints/tokenizer_init.pt','cuda:0')

# 2. Target codes from the frozen tokenizer, then the recognizer (same tokenizer throughout).
# code_agreement_check('./checkpoints/tokenizer_best.pt')
# precompute_target_codes('./checkpoints/tokenizer_best.pt')
# precompute_multi_target_codes('./checkpoints/tokenizer_best.pt')   # only when multi_ref = True
# train('recognizer','./checkpoints/tokenizer_best.pt','cuda:0')

# 3. Decoder retrained on the recognizer's predicted codes.
# !cp checkpoints/recognizer_best.pt checkpoints/recognizer_final.pt
# train('rec2mel','./checkpoints/recognizer_final.pt','cuda:0')
# then in jasper.ipynb: run_jasper("checkpoints/rec2mel_best.pt", vocode_original=False)

# Merged code classes (set merge_classes): fine-tune a relabelled copy of a trained recognizer, then step 3.
# !cp checkpoints/recognizer_final.pt checkpoints/recognizer_init.pt
# set_checkpoint_mode('checkpoints/recognizer_init.pt', 'tokenizer')
# train('recognizer','./checkpoints/recognizer_init.pt','cuda:0')

# Decoder-only alternative on a frozen tokenizer (only needed for a skip-trained tokenizer):
# train('codes2mel','./checkpoints/pretrain_best.pt','cuda:0')

In [ ]:
# train('vocoder_adapt','./checkpoints/e2e_best.pt','cuda:0')

